# SODE-Guard v2 re-run on ICS3D (Colab, disconnect-proof)

This notebook re-runs SODE-Guard with modules 1–3 over five seeds, and then the three extras, all from one released pipeline:

* **Core** (12 models × 5 seeds): `sode_base` (the rebuttal recipe), `sode`, `ode`, `mlp`, `lgbm`, `xgb`, `rf`, `sode_at`, `ode_at`, `mlp_at`, `sode_rs`, `mlp_rs`. Each seed runs training, the lite attack suite and smoothing certificates.
* **Extras:**
  * `fullcheck` (about 1–2 GPU hours): seed 0 re-attacked with the full-strength rebuttal attacks and certified with n = 10,000. This checks that the lite attacks used for all seeds are strong enough.
  * `ablations` (the longest extra): by default 10 ablation models × 5 seeds. These are post-hoc off or per-class, plain CE, focal, class-balanced CE, square-root sampling, quantile encoding, PGD-5 adversarial training, PGD-5 MLP and smoothed neural ODE. TRADES and SDE-TGNN can be added in `ABLATIONS`.
  * `classtime`: a per-class chronological split, on 3 seeds.

### How it survives disconnects
* The experiments run in a **background process** on the runtime, not inside a notebook cell. A browser disconnect or a stopped cell therefore does not stop them.
* The work happens on the runtime's local disk. After **every finished job**, and every 5 minutes, it is copied to **Google Drive** (`MyDrive/sode_guard_v2`).
* Every stage skips work that is already finished.

**If the runtime is reset or you get disconnected, choose *Runtime → Run all*.** The notebook restores from Drive, starts the background run again if it is not running, and carries on. At most the job that was in progress is lost.

### Before you start
* *Runtime → Change runtime type → GPU* (A100 or L4 if you have them; a T4 works but is slower).
* Keep this tab open while it runs. Free Colab ends idle sessions and caps a session at about 12 hours, so expect to press *Run all* again once or twice.
* A rough runtime guess on a single T4 is 4–8 hours for the core and about 15–20 hours more for the default extras. An A100 takes well under half that. The core results are aggregated first, so they are usable before the extras finish.
* On a runtime **without a GPU**, *Run all* restores the results and step 9 works, but the background run is not started.
* If a Kaggle run saved its output earlier, you can reuse it. Download that run's `v2` folder and put it in Drive at `MyDrive/sode_guard_v2/v2`, and this notebook resumes from it.

## 0. Settings

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/sode_guard_v2'   # persistent copy of everything
LOCAL = '/content/v2'                                  # fast working copy on the runtime
SEEDS = '0 1 2 3 4'
EXTRAS = ['fullcheck', 'ablations', 'classtime']      # run after the core, in this order; remove any you do not want
# ablation models (each runs on all seeds); the two most expensive and least important are left out
ABLATIONS = 'sode_noposthoc sode_perclass sode_ce sode_focal sode_cb sode_sampler sode_quant sode_at_pgd mlp_at_pgd ode_rs'
# full list adds: sode_at_trades sdetgnn
HOURS = 11.5        # stop cleanly after this many hours (free Colab: <= 11.5; Colab Pro: up to 23)
RUN_SMOKE = True    # 5-10 minute check before the first long run; skipped once results exist

## 1. GPU check

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 2. Code (re-running this cell keeps the code if the background run is active)

In [ ]:
import os, subprocess
def driver_running():
    return subprocess.run(['pgrep', '-f', 'scripts/v2_driver.py'], capture_output=True).returncode == 0
if not os.path.isdir('/content/CV/SODE-Guard'):
    !git clone -q --depth 1 -b claude/implement-ode-guard-A39oM https://github.com/rogerpanel/CV.git /content/CV
elif not driver_running():
    !cd /content/CV && git pull -q --ff-only
%cd /content/CV/SODE-Guard
!git log -1 --format='code commit: %h %ci'
!pip -q install kaggle kagglehub lightgbm xgboost scikit-learn scipy pandas

## 3. Google Drive and restore

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, shutil
os.makedirs(DRIVE_ROOT, exist_ok=True)
# results of the earlier Colab notebook (seed folders directly in DRIVE_ROOT) are reused too
if os.path.exists(f'{DRIVE_ROOT}/full.npz') and not os.path.exists(f'{DRIVE_ROOT}/v2/full.npz'):
    print('found results from the earlier notebook layout: moving them into', f'{DRIVE_ROOT}/v2')
    os.makedirs(f'{DRIVE_ROOT}/v2', exist_ok=True)
    for f in os.listdir(DRIVE_ROOT):
        if not f.startswith('v2'):
            shutil.move(f'{DRIVE_ROOT}/{f}', f'{DRIVE_ROOT}/v2/{f}')
    for suffix in ('_fullcheck', '_classtime'):
        old = DRIVE_ROOT + suffix
        if os.path.isdir(old) and not os.path.isdir(f'{DRIVE_ROOT}/v2{suffix}'):
            shutil.copytree(old, f'{DRIVE_ROOT}/v2{suffix}')
if not driver_running():     # copy Drive -> runtime, and set aside any file a dead session left unreadable
    !python scripts/v2_driver.py --out {LOCAL} --sync-to {DRIVE_ROOT} --restore
os.makedirs(LOCAL, exist_ok=True)
done = len(glob.glob(f'{LOCAL}/seed_*/*.pt')) + len(glob.glob(f'{LOCAL}/seed_*/*.pkl'))
print('trained models available locally:', done)

## 4. Data: Kaggle download (skipped when the prepared data is already in Drive)

The token is a Colab **secret** named `KAGGLE_API_TOKEN`: key icon in the left bar, with *Notebook access* on. Downloading through the API does not use any Kaggle GPU quota. Never paste the token into a code cell.

In [ ]:
CSV = '/content/ics3d/Containers_Dataset.csv'
DATASET = 'rogernickanaedevha/integrated-cloud-security-3datasets-ics3d'
if os.path.exists(f'{LOCAL}/full.npz'):
    print('prepared data present: no download needed')
elif not os.path.exists(CSV):
    from google.colab import userdata
    token = userdata.get('KAGGLE_API_TOKEN')
    assert token and token.startswith('KGAT_'), 'Add the KAGGLE_API_TOKEN secret first (key icon, left bar)'
    os.environ['KAGGLE_API_TOKEN'] = token
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(token)
    os.chmod('/root/.kaggle/access_token', 0o600)
    os.makedirs('/content/ics3d', exist_ok=True)
    r = subprocess.run(['kaggle', 'datasets', 'download', '-d', DATASET, '-f', 'Containers_Dataset.csv',
                        '-p', '/content/ics3d', '--unzip', '-q'], capture_output=True, text=True)
    for z in glob.glob('/content/ics3d/*.zip'):
        subprocess.run(['unzip', '-o', '-q', z, '-d', '/content/ics3d']); os.remove(z)
    if not os.path.exists(CSV):
        print('single-file download failed, using kagglehub:', r.stderr[-300:])
        import kagglehub
        path = kagglehub.dataset_download(DATASET)
        hits = glob.glob(f'{path}/**/Containers_Dataset.csv', recursive=True)
        assert hits, 'Containers_Dataset.csv not found in the download'
        os.symlink(hits[0], CSV)
    print(f'{os.path.getsize(CSV)/1e6:.0f} MB  {CSV}')

## 5. Prepare (cap 20,000 flows per class, drop identifiers and the source port, remove duplicates)

In [ ]:
if not os.path.exists(f'{LOCAL}/full.npz'):
    !python scripts/v2_run.py prepare --csv "{CSV}" --label-col Label --drop-cols "Src Port" --out {LOCAL} 2>&1 | tail -4
    !python scripts/v2_driver.py --out {LOCAL} --sync-to {DRIVE_ROOT} --sync-only
import json
meta = json.load(open(f'{LOCAL}/meta.json'))
print(json.dumps({k: meta[k] for k in ('classes', 'class_counts_after_dedup', 'dropped_small_classes', 'dedup')}, indent=1))

## 6. Smoke test (first run only, about 5–10 minutes; its numbers mean nothing)

In [ ]:
import time
if RUN_SMOKE and not glob.glob(f'{LOCAL}/seed_*/results.json') and not driver_running():
    TEST = '/content/v2_smoke'
    !rm -rf {TEST} && mkdir -p {TEST} && cp {LOCAL}/full.npz {LOCAL}/meta.json {TEST}/
    !python scripts/v2_driver.py --out {TEST} --seeds 0 --deadline {time.time() + 1800} --margin 60 --heartbeat 120 --extra-args "--epochs 1 --n-val-robust 200 --n-attack 100 --restarts 1 --long-steps 5 --eot 2 --eot-apgd 2 --apgd-iters 5 --eot-steps 5 --square-queries 20 --em-paths 16 --n-cert 50 --smooth-n 200 --sanity-seeds" | tail -16
else:
    print('skipped')

## 7. Start the background run

This starts the run only if it is not already going, so re-running the cell is safe. Its log is `v2/driver.log`, also on Drive.

In [ ]:
if driver_running():
    print('background run already active: go to step 8')
elif not torch.cuda.is_available():
    print('No GPU on this runtime: the background run was NOT started (training on CPU would take days).')
    print('Results already on Drive are restored; step 9 still works. Reconnect with a GPU to continue.')
else:
    deadline = time.time() + HOURS * 3600
    cmd = ['python', 'scripts/v2_driver.py', '--out', LOCAL, '--sync-to', DRIVE_ROOT, '--seeds', *SEEDS.split(),
           '--deadline', str(deadline), '--heartbeat', '300']
    if EXTRAS:
        cmd += ['--extras', *EXTRAS, '--ablations', *ABLATIONS.split()]
    log = open(f'{LOCAL}/driver.log', 'a')
    log.write(f'\n===== started {time.ctime()} =====\n'); log.flush()
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True,
                         cwd='/content/CV/SODE-Guard')
    print('started background run, pid', p.pid, '| stops by', time.ctime(deadline))

## 8. Monitor

Refreshes every 5 minutes until the run finishes. You can stop this cell at any time: the background run keeps going. After a disconnect, choose *Runtime → Run all* (or run steps 0–3 and this cell).

In [ ]:
from IPython.display import clear_output
def show():
    clear_output(wait=True)
    print(time.ctime(), '| background run active:', driver_running())
    pj = f'{LOCAL}/progress.json'
    if os.path.exists(pj):
        st = json.load(open(pj))['status']
        counts = {}
        for v in st.values():
            counts[v.split(' (')[0]] = counts.get(v.split(' (')[0], 0) + 1
        print('jobs:', counts)
        for k, v in st.items():
            if v != 'done':
                print(f'  {v[:50]:50s} {k}')
    lines = [l for l in open(f'{LOCAL}/driver.log', errors='ignore').read().splitlines() if l.strip()] if os.path.exists(f'{LOCAL}/driver.log') else []
    print('\n'.join(lines[-25:]))
    !nvidia-smi --query-gpu=utilization.gpu,memory.used,memory.total --format=csv,noheader
show()
while driver_running():
    time.sleep(300)
    show()
print('\nbackground run finished')

## 9. Results

Shows `summary.md` for the core and ablations, and for the full-strength check and the per-class time split when they exist. It writes `v2_results.zip` (no model weights) to Drive and downloads it. Upload that zip to the chat.

In [ ]:
import zipfile
for name in ('v2', 'v2_fullcheck', 'v2_classtime'):
    f = f'/content/{name}/summary.md'
    if os.path.exists(f):
        print(f'\n######## {name}\n'); print(open(f).read())
zp = f'{DRIVE_ROOT}/v2_results.zip'
keep = ('.md', '.json', '.log', '.npz')
with zipfile.ZipFile('/content/v2_results.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('v2', 'v2_fullcheck', 'v2_classtime'):
        for f in glob.glob(f'/content/{name}/**/*', recursive=True):
            if f.endswith(keep) and os.path.basename(f) not in ('full.npz', 'data.npz') and '.tmp' not in f:
                z.write(f, os.path.relpath(f, '/content'))
shutil.copy2('/content/v2_results.zip', zp)
print('\nv2_results.zip:', round(os.path.getsize(zp) / 1e6, 1), 'MB, also saved to', zp)
from google.colab import files
files.download('/content/v2_results.zip')

## 10. (Only if needed) stop the background run

In [ ]:
STOP = False   # set to True and run this cell to stop everything; finished work is already on Drive
if STOP:
    !pkill -f scripts/v2_driver.py; pkill -f scripts/v2_run.py
    !python scripts/v2_driver.py --out {LOCAL} --sync-to {DRIVE_ROOT} --sync-only
    print('stopped; running background:', driver_running())

## 11. (Optional) Continue on Kaggle: upload the saved results as a private Kaggle dataset

Works on a runtime **without a GPU**: run steps 0, 2 and 3 first, then this cell. It needs the `KAGGLE_API_TOKEN` secret and your Kaggle **username** (shown on your Kaggle profile). It zips everything in `MyDrive/sode_guard_v2` (models included) and uploads it as the private dataset `<username>/sode-guard-v2-state`; running it again adds a new version. On Kaggle, add that dataset to `v2_rerun_ics3d_kaggle.ipynb` and commit.

In [ ]:
KAGGLE_USER = ''      # <- your Kaggle username
assert KAGGLE_USER, 'set KAGGLE_USER to your Kaggle username'
from google.colab import userdata
token = userdata.get('KAGGLE_API_TOKEN')
assert token and token.startswith('KGAT_'), 'Add the KAGGLE_API_TOKEN secret first (key icon, left bar)'
os.environ['KAGGLE_API_TOKEN'] = token
os.makedirs('/root/.kaggle', exist_ok=True)
open('/root/.kaggle/access_token', 'w').write(token); os.chmod('/root/.kaggle/access_token', 0o600)
import zipfile, json, time, glob, shutil, subprocess
EXP = '/content/kaggle_state'
shutil.rmtree(EXP, ignore_errors=True); os.makedirs(EXP)
with zipfile.ZipFile(f'{EXP}/sode_guard_v2_state.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for name in ('v2', 'v2_fullcheck', 'v2_classtime'):
        for f in glob.glob(f'/content/{name}/**/*', recursive=True):
            if os.path.isfile(f) and '.tmp' not in f and not f.endswith('.corrupt'):
                z.write(f, os.path.relpath(f, '/content'))
print('state zip:', round(os.path.getsize(f'{EXP}/sode_guard_v2_state.zip') / 1e6), 'MB')
slug = f'{KAGGLE_USER}/sode-guard-v2-state'
json.dump({'title': 'sode-guard-v2-state', 'id': slug, 'licenses': [{'name': 'CC0-1.0'}]},
          open(f'{EXP}/dataset-metadata.json', 'w'))
exists = subprocess.run(['kaggle', 'datasets', 'status', slug], capture_output=True, text=True).returncode == 0
cmd = (['kaggle', 'datasets', 'version', '-p', EXP, '-m', 'state ' + time.ctime()] if exists
       else ['kaggle', 'datasets', 'create', '-p', EXP])
r = subprocess.run(cmd, capture_output=True, text=True)
print(r.stdout[-800:], r.stderr[-800:])
print('Private dataset:', f'https://www.kaggle.com/datasets/{slug}', '(it can take a few minutes to finish processing)')